# 08 — Finite-$N$ (Discrete) Effects

Convergence of the discrete population sum to the continuous integral; bifurcation locus shift at biological $N=8$; symmetry preservation at finite $N$.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

from steering.params import ModelParams, ForcingParams
from steering.models import (
    DuffingModel,
    BesselSteeringModel,
    ContinuousPFLModel,
    DiscretePFLModel,
    FullCircuitModel,
)
from steering.dynamics import VelocityDynamics, AccelerationDynamics
from steering.integrator import Simulation
from steering.visualization.style import use_paper_style

use_paper_style()


In [ ]:
cont = ContinuousPFLModel(n_quad=256)
disc = DiscretePFLModel()
p = ModelParams(kappa_h=2.0, kappa_g=2.0, delta=1.4)
theta = np.linspace(-np.pi, np.pi, 401)
U_inf = cont.steering_drive(theta, p)


## $L^\infty$ convergence of $U(\theta)$

In [ ]:
Ns = [4, 8, 16, 32, 64, 256]
errs = []
for N in Ns:
    U_N = disc.steering_drive(theta, p.replace(N_neurons=N))
    errs.append(np.max(np.abs(U_N - U_inf)))
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.loglog(Ns, errs, 'o-')
ax.set_xlabel('N'); ax.set_ylabel(r'$\|U_N - U_\infty\|_\infty$')
plt.show()


## Bifurcation locus at finite $N$

In [ ]:
from scipy.optimize import brentq
from steering.models import BesselSteeringModel
bessel = BesselSteeringModel()

def find_pitchfork(model, kappa, get_U_at_theta):
    def c1_at(d):
        pp = ModelParams(kappa_h=kappa, kappa_g=kappa, delta=d)
        h = 1e-3
        return (get_U_at_theta(h, pp) - get_U_at_theta(-h, pp)) / (2*h)
    try:
        return brentq(c1_at, 0.5, np.pi/2 - 1e-3)
    except ValueError:
        return float('nan')

kappas = np.linspace(0.5, 5, 12)
curves = {}
for N in [None, 8, 16, 64]:
    ds = []
    for k in kappas:
        if N is None:
            ds.append(find_pitchfork(bessel, k,
                lambda th, pp: float(np.asarray(bessel.steering_drive(th, pp)))))
        else:
            ds.append(find_pitchfork(disc, k,
                lambda th, pp, _N=N: float(disc.steering_drive(th, pp.replace(N_neurons=_N)))))
    curves[f'N={N}' if N is not None else 'continuous'] = ds
fig, ax = plt.subplots(figsize=(6, 4))
for label, ds in curves.items():
    ax.plot(kappas, ds, label=label)
ax.set_xlabel(r'$\kappa$'); ax.set_ylabel(r'$\delta_{\mathrm{pitchfork}}$')
ax.legend(); plt.show()
